In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
from tqdm import tqdm

def brier_score_loss(
    y_true,
    y_proba,
    *,
    sample_weight=None,
    pos_label=None,
    labels=None,
    scale_by_half="auto",
):
    label2idx = {label: i for i, label in enumerate(labels)}
    rows, cols = zip(*[(i, label2idx[el]) for i, el in enumerate(y_true)])
    y_proba_true = np.zeros_like(y_proba)
    y_proba_true[rows, cols] = 1

    brier_score = np.average(
        np.sum((y_proba_true - y_proba) ** 2, axis=1), weights=sample_weight
    )

    if scale_by_half == "auto":
        scale_by_half = y_proba.ndim == 1 or y_proba.shape[1] < 3
    if scale_by_half:
        brier_score *= 0.5

    return float(brier_score)

In [ ]:
df = pd.read_excel('/data/users/quentin/final_package/experiments/survival_analysis/data/Melanoma_clinical_data_jack_may7.xlsx', '20211001_Melanoma_clinical_data')


In [ ]:
for col in df.columns:
    if "immu" in col:
        print(col)

In [ ]:
df = pd.read_csv('ord_longi.csv', index_col=0)

df["metastasis_location"] = df["metastasis_location"].apply(lambda x: ("lymph node" in x))

In [ ]:
pd.factorize(df["stage"])

In [ ]:
df["stage"] = pd.factorize(df["stage"])[0]
df["subtype"] = pd.factorize(df["subtype"])[0]

In [ ]:
def convert_state(state):
    if state in [1,2]:
        return 1
    elif state in [3,4]:
        return 2
    elif state in [5]:
        return 3
    elif state in [6]:
        return 4
    else:
        assert False
        
def convert_state(state):
    if state in [1,2,3]:
        return 1
    elif state in [4,5,6]:
        return 0
    else:
        assert False
    
df['state'] = df['state'].apply(convert_state)
df['prev_state'] = df['prev_state'].apply(convert_state)

In [ ]:
df.head()

In [ ]:
#print(list(df.columns))
columns2keep = ['age', 'sex', "brain_metastasis", "metastasis_location", "stage", "subtype", 'fraction_WP1_NM_norm_long', 'resistant_WP1_NM_norm_long']#,'resistant_WP1_NM_norm','score_WP1_NM_norm_long','score_WP1_NM_norm','fraction_WP1_NM_norm_long','fraction_WP1_NM_norm']
columns2keep_FD = ['age', 'sex', "brain_metastasis", "metastasis_location","stage", "subtype",   'score_FD_NM_norm_long']
columns2keep_FM = ['age', 'sex', "brain_metastasis", "metastasis_location", "stage", "subtype", 'score_FM_NM_norm_long', 'resistant_FM_NM_norm_long']
columns2keep_baseline = ['age', 'sex', "brain_metastasis", "metastasis_location"]

immuno_feat = "immunotherapies_tot_after_biopsy"
chemo_feat = "chemotherapies_tot_after_biopsy"
columns2keep.append(immuno_feat)
columns2keep_FD.append(immuno_feat)
columns2keep_FM.append(immuno_feat)
columns2keep_baseline.append(immuno_feat)


columns2keep.append(chemo_feat)
columns2keep_FD.append(chemo_feat)
columns2keep_FM.append(chemo_feat)
columns2keep_baseline.append(chemo_feat)



In [ ]:
tuproIDs = df['tupro_id'].unique()
has_used_TD = {TID:False for TID in tuproIDs}
for itupro, tuproID in enumerate(tuproIDs):
    dffiltered = df.loc[df['tupro_id']==tuproID]
    if np.sum(dffiltered['targeted_tot_after_biopsy'])>1:
        has_used_TD[tuproID] = True

In [ ]:
alldays = df['days'].unique()
states = df['state'].unique()
print(alldays, states)

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.inspection import permutation_importance
# ============================
# Model definition (stabilized RF)
# ============================

def create_model():
    return RandomForestClassifier(
        n_estimators=500,
        max_depth=4,
        min_samples_leaf=3,
        max_features="sqrt",
        class_weight="balanced",
        bootstrap=True,
        random_state=0,
        n_jobs=-1
    )



# ============================
# Containers
# ============================

probs_dict = {'ours': [], 'baseline': [], 'FD': [], 'FM': []}
y_true_dict = {'ours': [], 'baseline': [], 'FD': [], 'FM': []}

feature_importances = {
    'ours': [],
    'baseline': [],
    'FD': [],
    'FM': []
}


# ============================
# Filter days with enough samples
# ============================

alldays = alldays[[i for i, days in enumerate(alldays) if np.sum(df['days']==days) > 2]]
print("Days used:", alldays)

logo = LeaveOneGroupOut()


# ============================
# Main Loop
# ============================

for days in tqdm(alldays):

    df_filtered = df[df['days'] == days]

    y = df_filtered['state'].to_numpy().astype(int)
    groups = df_filtered['tupro_id'].to_numpy()

    datasets = {
        "ours": df_filtered[columns2keep].to_numpy(),
        "baseline": df_filtered[columns2keep_baseline].to_numpy(),
        "FD": df_filtered[columns2keep_FD].to_numpy(),
        "FM": df_filtered[columns2keep_FM].to_numpy()
    }

    # run CV
    for train_idx, test_idx in logo.split(datasets["ours"], y, groups):
        
        if has_used_TD[groups[test_idx[0]]]:

            for model_name, X in datasets.items():

                X_train = X[train_idx]
                X_test = X[test_idx]

                y_train = y[train_idx]
                y_test = y[test_idx]

                clf = create_model()
                clf.fit(X_train, y_train)

                prob = clf.predict_proba(X_test)[:,1]
                pred = clf.predict(X_test)

                probs_dict[model_name].append(prob[0])
                y_true_dict[model_name].append(y_test[0])

                feature_importances[model_name].append(clf.feature_importances_)


# ============================
# Final Evaluation
# ============================

results = {}

for model_name in probs_dict.keys():

    y_true = np.array(y_true_dict[model_name])
    probs = np.array(probs_dict[model_name])

    preds = (probs > 0.5).astype(int)

    acc = accuracy_score(y_true, preds)

    try:
        auc = roc_auc_score(y_true, probs)
    except:
        auc = np.nan

    results[model_name] = {
        "accuracy": acc,
        "roc_auc": auc
    }


print("\n===== FINAL RESULTS =====")

for model, res in results.items():
    print(f"{model}: ACC={res['accuracy']:.3f}  AUC={res['roc_auc']:.3f}")


# ============================
# Feature Stability Analysis
# ============================

print("\n===== FEATURE IMPORTANCE STABILITY =====")

for model_name, importances in feature_importances.items():

    mean_importance = np.mean(importances, axis=0)
    std_importance = np.std(importances, axis=0)

    print(f"\nModel: {model_name}")

    if model_name == "ours":
        cols = columDear Dean,

    elif model_name == "baseline":
        cols = columns2keep_baseline
    elif model_name == "FD":
        cols = columns2keep_FD
    else:
        cols = columns2keep_FM

    for c, m, s in zip(cols, mean_importance, std_importance):
        print(f"{c:35s} mean={m:.4f}  std={s:.4f}")

# LOO

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import numpy as np
from tqdm import tqdm

## LEAVE ON OUT

# Containers
clfs = []
acc_scores, acc_scores_baseline, acc_scores_FD, acc_scores_FM = [], [], [], []
probs_dict = { 'ours': [], 'baseline': [], 'FD': [], 'FM': [] }
y_true_dict = { 'ours': [], 'baseline': [], 'FD': [], 'FM': [] }
phase2IDs = []

alldays = alldays[[i for i, days in enumerate(alldays) if np.sum(df['days']==days) > 2]]
print(alldays)

for i, days in tqdm(enumerate(alldays)):
    acc_scores.append([]); acc_scores_baseline.append([])
    acc_scores_FD.append([]); acc_scores_FM.append([])

    df_filtered = df[df['days'] == days]
    X = df_filtered[columns2keep].to_numpy()
    X_baseline = df_filtered[columns2keep_baseline].to_numpy()
    X_FD = df_filtered[columns2keep_FD].to_numpy()
    X_FM = df_filtered[columns2keep_FM].to_numpy()
    y = df_filtered['state'].to_numpy().astype(int)
    
    filtered_IDs = df_filtered['tupro_id'].unique()
    phase2IDs.append([])

    for tupro_id in filtered_IDs:  
        idxs = (df_filtered['tupro_id'] != tupro_id).to_numpy().astype(bool)

        # --- helper function to train/evaluate one model ---
        def evaluate_model(X_train, X_test, model_name):
            clf = RandomForestClassifier(max_depth=6, random_state=0)
            clf.fit(X_train[idxs], y[idxs])
            preds = clf.predict(X_train[~idxs])
            probs = clf.predict_proba(X_train[~idxs])[:, 1]
            acc = accuracy_score(y[~idxs], preds)
            clfs.append(clf)
            probs_dict[model_name].append(probs[0])  # one test sample each
            y_true_dict[model_name].append(y[~idxs][0])
            return acc

        acc_scores[i].append(evaluate_model(X, X, 'ours'))
        acc_scores_baseline[i].append(evaluate_model(X_baseline, X_baseline, 'baseline'))
        acc_scores_FD[i].append(evaluate_model(X_FD, X_FD, 'FD'))
        acc_scores_FM[i].append(evaluate_model(X_FM, X_FM, 'FM'))


In [ ]:
results = {
    "probs_dict": probs_dict,
    "y_true_dict": y_true_dict,
    #"acc_scores": acc_scores,
#     "acc_scores_baseline": acc_scores_baseline,
#     "acc_scores_FD": acc_scores_FD,
#     "acc_scores_FM": acc_scores_FM,
}

In [ ]:
from MLstatkit import Delong_test
models = ['ours', 'FM', 'baseline', 'FD']

for i in range(len(models)):
    for j in range(i+1, len(models)):
        m1, m2 = models[i], models[j]

        y_true = np.array(results['y_true_dict'][m1])
        pred1 = np.array(results['probs_dict'][m1])
        pred2 = np.array(results['probs_dict'][m2])
        np.all(results['y_true_dict'][m1] == results['y_true_dict'][m2])

        z, p = Delong_test(y_true, pred1, pred2, return_ci=False, return_auc=False)

        print("{0} vs {1}  p = {2}".format(m1,m2,p))
        print('\n')

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt
import numpy as np

# Load (if needed)
# with open("model_results.pkl", "rb") as f:
#     results = pickle.load(f)


name2label = {"ours": "scClone2DR",
             "baseline": "Baseline",
             "FD": "scFPM",
             "FM": "Clonal FM"}
plt.figure(figsize=(7,6))
for model_name, color, linestyle in zip(['ours', 'baseline', 'FD', 'FM'],
                             ['#009E73', '#B3A12F', 'red', '#0072B2'],
                             ['-', '--', '-.', ':']):
    y_true = np.array(results['y_true_dict'][model_name])
    probs = np.array(results['probs_dict'][model_name])
    fpr, tpr, _ = roc_curve(y_true, probs)
    auc = roc_auc_score(y_true, probs)
    plt.plot(fpr, tpr, label=f"{name2label[model_name]} (AUC={auc:.2f})", color=color, linestyle=linestyle, linewidth=2.2)

plt.plot([0, 1], [0, 1], label="Chance", color="black")
plt.xlabel("False Positive Rate", fontsize=16)
plt.ylabel("True Positive Rate", fontsize=16)
#plt.title("ROC Curves for Models")
plt.legend(fontsize=15)
plt.tight_layout()
plt.savefig('/data/users/quentin/package_paper/experiments/survival_analysis/ROCs_regu.png', dpi=200)

plt.show()

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt
import numpy as np

# Load (if needed)
# with open("model_results.pkl", "rb") as f:
#     results = pickle.load(f)


name2label = {"ours": "scClone2DR",
             "baseline": "Baseline",
             "FD": "scFPM",
             "FM": "Clonal FM"}
plt.figure(figsize=(7,6))
for model_name, color, linestyle in zip(['ours', 'baseline', 'FD', 'FM'],
                             ['#009E73', '#B3A12F', 'red', '#0072B2'],
                             ['-', '--', '-.', ':']):
    y_true = np.array(results['y_true_dict'][model_name])
    probs = np.array(results['probs_dict'][model_name])
    fpr, tpr, _ = roc_curve(y_true, probs)
    auc = roc_auc_score(y_true, probs)
    plt.plot(fpr, tpr, label=f"{name2label[model_name]} (AUC={auc:.2f})", color=color, linestyle=linestyle, linewidth=2.2)

plt.plot([0, 1], [0, 1], label="Chance", color="black")
plt.xlabel("False Positive Rate", fontsize=16)
plt.ylabel("True Positive Rate", fontsize=16)
#plt.title("ROC Curves for Models")
plt.legend(fontsize=15)
plt.tight_layout()
plt.savefig('/data/users/quentin/package_paper/experiments/survival_analysis/ROCs.png', dpi=200)

plt.show()


In [ ]:



alldays = alldays[[i for i, days in enumerate(alldays) if np.sum(df['days']==days) > 2]]
print(alldays)

for i, days in tqdm(enumerate(alldays)):
    # Containers
    clfs = []
    acc_scores, acc_scores_baseline, acc_scores_FD, acc_scores_FM = [], [], [], []
    probs_dict = { 'ours': [], 'baseline': [], 'FD': [], 'FM': [] }
    y_true_dict = { 'ours': [], 'baseline': [], 'FD': [], 'FM': [] }
    phase2IDs = []

    acc_scores.append([]); acc_scores_baseline.append([])
    acc_scores_FD.append([]); acc_scores_FM.append([])

    df_filtered = df[df['days'] == days]
    X = df_filtered[columns2keep].to_numpy()
    X_baseline = df_filtered[columns2keep_baseline].to_numpy()
    X_FD = df_filtered[columns2keep_FD].to_numpy()
    X_FM = df_filtered[columns2keep_FM].to_numpy()
    y = df_filtered['state'].to_numpy().astype(int)
    
    filtered_IDs = df_filtered['tupro_id'].unique()
    phase2IDs.append([])

    for tupro_id in filtered_IDs:  
        idxs = (df_filtered['tupro_id'] != tupro_id).to_numpy().astype(bool)

        # --- helper function to train/evaluate one model ---
        def evaluate_model(X_train, X_test, model_name):
            clf = RandomForestClassifier(max_depth=6, random_state=0)
            clf.fit(X_train[idxs], y[idxs])
            preds = clf.predict(X_train[~idxs])
            probs = clf.predict_proba(X_train[~idxs])[:, 1]
            acc = accuracy_score(y[~idxs], preds)
            clfs.append(clf)
            probs_dict[model_name].append(probs[0])  # one test sample each
            y_true_dict[model_name].append(y[~idxs][0])

            
        (evaluate_model(X, X, 'ours'))
        (evaluate_model(X_baseline, X_baseline, 'baseline'))
        (evaluate_model(X_FD, X_FD, 'FD'))
        (evaluate_model(X_FM, X_FM, 'FM'))
    results = {
    "probs_dict": probs_dict,
    "y_true_dict": y_true_dict
    }
    from sklearn.metrics import roc_curve, roc_auc_score
    import matplotlib.pyplot as plt
    import numpy as np

    # Load (if needed)
    # with open("model_results.pkl", "rb") as f:
    #     results = pickle.load(f)

    plt.figure(figsize=(7,6))
    for model_name, color in zip(['ours', 'baseline', 'FD', 'FM'],
                                 ['blue', 'orange', 'red', 'green']):
        y_true = np.array(results['y_true_dict'][model_name])
        probs = np.array(results['probs_dict'][model_name])
        fpr, tpr, _ = roc_curve(y_true, probs)
        auc = roc_auc_score(y_true, probs)
        plt.plot(fpr, tpr, label=f"{model_name} (AUC={auc:.2f})", color=color)

    plt.plot([0, 1], [0, 1], 'k--', label="Chance")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curves for Models")
    plt.legend()
    plt.tight_layout()
    plt.savefig("/data/users/quentin/final_package/experiments/survival_analysis/figures/ROCs_FUP{0}.png".format(i), dpi=200)

    plt.show()


In [ ]:
probs_dict